# Memory experiment - rotated surface code with a data-qubit defect

Build and visualize a rotated surface-code memory experiment in which the center data qubit is measured and disabled. Neighboring checks become alternating X- and Z-type gauge measurements while unaffected checks remain active. Batch LER sweeps use [`benchmarks/memory/run_memory.py`](../../benchmarks/memory/run_memory.py).

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt

from lightstim.noise.config import NoiseConfig
from lightstim.plot.styles import PALETTE, apply_paper_style, bold_ticks
from lightstim.protocols import MemoryExperiment
from lightstim.protocols.rotated_surface_defect import (
    RotatedSurfaceDefectMemoryExperiment,
)
from lightstim.qec_code.surface_code.rotated import (
    RotatedSurfaceCode,
    RotatedSurfaceCodeExtractionBlock,
)
from lightstim.simulation.decoder_backend import DecoderConfig, SimulationPipeline

## Build the experiment

One pre-defect round and four post-defect gauge rounds show two complete Z/X alternations. The number of post-defect rounds is the length of `post_defect_schedule`.

In [ ]:
experiment = RotatedSurfaceDefectMemoryExperiment(
    distance=3,
    pre_defect_rounds=1,
    post_defect_schedule=("Z", "X", "Z", "X"),
)
circuit = experiment.build().without_noise()

print("defect coordinate:", experiment.defect_coord)
print("gauge schedule:", experiment.gauge_schedule)
print(
    f"qubits={circuit.num_qubits}, "
    f"detectors={circuit.num_detectors}, "
    f"observables={circuit.num_observables}"
)

## Detector-slice visualization

The diagram includes the ordinary checks before the defect, the center-qubit readout, four alternating gauge rounds, and the final memory readout.

In [ ]:
circuit.diagram("detslice-with-ops-svg")

## LER scaling with and without the defect

For a fair `d=3` comparison, both circuits contain seven noisy syndrome-extraction rounds. The defect circuit uses three ordinary rounds followed by the same four-round gauge schedule shown above; its optional noiseless preparation round is disabled.

In [ ]:
distance = 3
p_values = (1e-3, 2e-3, 3e-3, 5e-3)
post_defect_schedule = ("Z", "X", "Z", "X")
shots = 20_000

pipeline = SimulationPipeline(
    decoder_config=DecoderConfig("pymatching", backend="cpu"),
    max_shots=shots,
    max_errors=shots + 1,
    batch_size=shots,
    num_workers=1,
    print_progress=False,
)
logical_error_rates = {"No defect": [], "One data defect": []}

for p in p_values:
    noise = NoiseConfig(
        p_idle=p, p_1q=p, p_2q=p, p_meas=p, p_reset=p
    )
    circuits = {
        "No defect": MemoryExperiment(
            qec_patch=RotatedSurfaceCode(distance=distance),
            extraction_block_class=RotatedSurfaceCodeExtractionBlock,
            rounds=distance + len(post_defect_schedule),
            noise_params=noise,
            basis="Z",
        ).build(),
        "One data defect": RotatedSurfaceDefectMemoryExperiment(
            distance=distance,
            preparation_rounds=0,
            pre_defect_rounds=distance,
            post_defect_schedule=post_defect_schedule,
            noise_params=noise,
        ).build(),
    }
    for label, noisy_circuit in circuits.items():
        stats = pipeline.run(noisy_circuit)
        logical_error_rates[label].append(stats.logical_error_rate)
        print(
            f"{label:15s} p={p:g}: "
            f"LER={stats.logical_error_rate:.3e} "
            f"({stats.errors}/{stats.shots:,})"
        )

apply_paper_style()
fig, ax = plt.subplots(figsize=(5.6, 4.0))
for label, color, marker in (
    ("No defect", PALETTE[1], "o"),
    ("One data defect", PALETTE[0], "s"),
):
    ax.loglog(
        p_values,
        logical_error_rates[label],
        color=color,
        marker=marker,
        label=label,
    )

ax.set_xlabel("Physical error rate $p$")
ax.set_ylabel("Logical error rate")
ax.set_title("$d=3$ rotated surface-code memory")
ax.set_xticks(p_values, labels=("0.001", "0.002", "0.003", "0.005"))
ax.tick_params(axis="x", which="minor", labelbottom=False)
ax.legend()
bold_ticks(ax)
fig.tight_layout(pad=0.4)
plt.show()